# Titanic exploration (Seaborn)

This notebook explores the Titanic dataset to answer: overall survival rate, effects of sex and class, relationship between age and survival, and interactions between sex/class/age. Each plot includes a short explanation. Plots are saved to the `plots/` folder.

In [1]:
# Setup: imports and load data
import os
import pandas as pd
import seaborn as sns
import matplotlib
# Use a non-interactive backend so nbconvert/headless runs don't fail
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

sns.set(style="whitegrid", palette="pastel")
# adjust this path if needed
path = r"c:\Users\AVADHESH KUMAR\Desktop\WYK DOCS\git\seaborn-data\seaborn-data\titanic.csv"
df = pd.read_csv(path)
# ensure plots directory exists
os.makedirs('plots', exist_ok=True)
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
# Basic info and cleaning

df.info()
display(df.describe(include='all'))
# Clean: convert types and fill missing ages
df['survived'] = df['survived'].astype(int)
df['sex'] = df['sex'].astype('category')
df['pclass'] = df['pclass'].astype('category')
df['age_missing'] = df['age'].isna()
# Fill missing age with median (not ideal but sufficient for visualization)
df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].astype(float)
df.sample(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
count,891.000000,891.000000,891,714.000000,891.000000,891.000000,891.000000,889,891,891,891,203,889,891,891
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,7,3,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,C,Southampton,no,True
freq,NaN,NaN,577,NaN,NaN,NaN,NaN,644,491,537,537,59,644,549,537
mean,0.383838,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.486592,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,age_missing
170,0,1,male,61.0,0,0,33.5000,S,First,man,True,B,Southampton,no,True,False
396,0,3,female,31.0,0,0,7.8542,S,Third,woman,False,NaN,Southampton,no,True,False
677,1,3,female,18.0,0,0,9.8417,S,Third,woman,False,NaN,Southampton,yes,True,False


## 1) Overall survival count

This bar plot shows counts of passengers who died vs survived. It answers: what was the survival rate overall?

In [3]:
plt.figure(figsize=(6,4))
sns.countplot(x='survived', data=df, palette=['#d9534f','#5cb85c'])
plt.xticks([0,1], ['Died','Survived'])
plt.title('Titanic: Overall Survival Count')
plt.xlabel('')
plt.ylabel('Count')
plt.savefig('plots/overall_survival.png', bbox_inches='tight')
plt.show()

C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\1108605657.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='survived', data=df, palette=['#d9534f','#5cb85c'])
C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\1108605657.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2) Survival rate by sex

Shows how survival depended on passenger sex. We plot average survival (rate) per sex to visualize difference between male and female outcomes.

In [4]:
plt.figure(figsize=(6,4))
sns.barplot(x='sex', y='survived', data=df, ci=None, palette='pastel')
plt.ylabel('Survival rate')
plt.ylim(0,1)
plt.title('Survival Rate by Sex')
plt.savefig('plots/survival_by_sex.png', bbox_inches='tight')
plt.show()

C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\2238754232.py:2: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x='sex', y='survived', data=df, ci=None, palette='pastel')
C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\2238754232.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='sex', y='survived', data=df, ci=None, palette='pastel')
C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\2238754232.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3) Survival rate by class

Shows how passenger class (1st, 2nd, 3rd) affected survival. We compute and plot average survival per class.

In [5]:
plt.figure(figsize=(7,4))
sns.barplot(x='pclass', y='survived', data=df, order=[1,2,3], palette='Blues')
plt.xlabel('Passenger Class')
plt.ylabel('Survival rate')
plt.title('Survival Rate by Class')
plt.ylim(0,1)
plt.savefig('plots/survival_by_class.png', bbox_inches='tight')
plt.show()

C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\1357859006.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='pclass', y='survived', data=df, order=[1,2,3], palette='Blues')


C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\1357859006.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4) Age distribution by survival and logistic fit

We examine the distribution of ages for survivors and non-survivors and fit a logistic curve to visualize probability of survival vs age.

In [6]:
plt.figure(figsize=(8,5))
sns.violinplot(x='survived', y='age', data=df, palette=['#d9534f','#5cb85c'])
plt.xticks([0,1], ['Died','Survived'])
plt.title('Age Distribution by Survival')
plt.savefig('plots/age_by_survival.png', bbox_inches='tight')
plt.show()

# Logistic fit: probability of survival vs age
# Try seaborn's logistic lmplot (requires statsmodels). If not available, fall back to sklearn.
try:
    g = sns.lmplot(x='age', y='survived', data=df, logistic=True, ci=None, height=5, aspect=1.3)
    # lmplot returns a FacetGrid; set title on the figure
    g.fig.suptitle('Probability of Survival vs Age (logistic fit)')
    g.fig.tight_layout(rect=[0,0,1,0.95])
    g.fig.savefig('plots/survival_vs_age_logistic.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    # Fallback: fit logistic regression with sklearn and plot predicted probability
    from sklearn.linear_model import LogisticRegression
    X = df[['age']].astype(float).values
    y = df['survived'].values
    model = LogisticRegression(solver='lbfgs', max_iter=200)
    model.fit(X, y)
    age_grid = np.linspace(X.min(), X.max(), 200).reshape(-1,1)
    probs = model.predict_proba(age_grid)[:,1]

    plt.figure(figsize=(7,5))
    sns.scatterplot(x='age', y='survived', data=df, alpha=0.25)
    plt.plot(age_grid.ravel(), probs, color='red')
    plt.ylim(-0.05,1.05)
    plt.title('Probability of Survival vs Age (logistic fit — sklearn fallback)')
    plt.xlabel('Age')
    plt.ylabel('Survival probability')
    plt.savefig('plots/survival_vs_age_logistic.png', bbox_inches='tight')
    plt.show()
    print(f'Fallback used for logistic fit due to: {e}')

C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\3864040169.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='survived', y='age', data=df, palette=['#d9534f','#5cb85c'])


C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\3864040169.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\3864040169.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5) Interaction: Class and Sex (grouped)

Shows survival rate broken down by class and sex to examine interactions (e.g., did women in 3rd class have different outcomes than men in 1st class?).

In [7]:
g = sns.catplot(x='pclass', y='survived', hue='sex', data=df, kind='bar', 
                order=[1,2,3], palette='Set2', height=5, aspect=1.2)
g.set_axis_labels('Passenger class','Survival rate')
plt.ylim(0,1)
plt.title('Survival Rate by Class and Sex')
plt.savefig('plots/survival_class_sex.png', bbox_inches='tight')
plt.show()

C:\Users\AVADHESH KUMAR\AppData\Local\Temp\ipykernel_31836\4125236939.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Short summary and next steps

- Early observations: females had substantially higher survival rates; 1st class had higher survival; younger passengers had somewhat higher survival probability.
- Next: perform formal modelling (logistic regression), compute confidence intervals, and add more annotation/Canva slides for presentation.
- Files saved: the notebook, the `plots/` PNGs, and `deliverable.md` (created alongside).